# Fine-tuning — YOLO26n com Data Augmentation

Experimento de detecção de `pothole` e `manhole_cover`. Esta versão aplica as correções de auditoria, utiliza o modelo Ultralytics YOLO26n e adiciona augmentation moderado no treinamento.

O notebook registra execuções, curvas e métricas. Não gera painéis de investigação ou comparações visuais aleatórias.

## 0. Dependências

In [ ]:
%pip install -U ultralytics pyyaml pandas matplotlib opencv-python

## 1. Ambiente e reprodutibilidade

In [ ]:
import os
import sys
import json
import csv
import random
import platform
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import torch
import torchvision
import ultralytics
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Ultralytics:", ultralytics.__version__)
print("Sistema:", platform.platform())
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Caminhos e identificação do experimento

In [ ]:
# False reutiliza a versão mais recente deste experimento (permite retomada).
# True cria uma nova versão, preservando as execuções anteriores.
NEW_VERSION = False

cwd = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [cwd, *cwd.parents] if p.name.lower() == "urbansense"),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Não encontrei a pasta 'urbansense' a partir de {cwd}. "
        "Defina PROJECT_ROOT manualmente."
    )

DATASET_DIR = PROJECT_ROOT / "data" / "Potholes_dataset_dividido"
DATA_YAML = DATASET_DIR / "data.yaml"
if not DATA_YAML.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {DATA_YAML}")

RUNS_DIR = PROJECT_ROOT / "runs"
ARTIFACTS_ROOT = PROJECT_ROOT / "artifacts"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_PREFIX = "potholes-yolo26n-augmentation"
existing = [
    p for p in RUNS_DIR.glob(f"{EXPERIMENT_PREFIX}-v*") if p.is_dir()
]
versions = [
    int(p.name.rsplit("-v", 1)[1])
    for p in existing
    if p.name.rsplit("-v", 1)[1].isdigit()
]
VERSION = max(versions, default=0) + 1 if NEW_VERSION else max(versions, default=1)

EXPERIMENT_NAME = f"{EXPERIMENT_PREFIX}-v{VERSION}"
RUN_NAME = EXPERIMENT_NAME
RUN_DIR = RUNS_DIR / RUN_NAME
ARTIFACTS_DIR = ARTIFACTS_ROOT / EXPERIMENT_NAME
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("Experimento:", EXPERIMENT_NAME)
print("Dataset:", DATASET_DIR)
print("Diretório de execução:", RUN_DIR)

## 3. Auditoria de integridade e anotações

In [ ]:
config = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
class_names_raw = config.get("names", {})
if isinstance(class_names_raw, list):
    class_names = {i: name for i, name in enumerate(class_names_raw)}
else:
    class_names = {int(k): v for k, v in class_names_raw.items()}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
dataset_rows = []
class_distribution = {int(k): 0 for k in class_names}
annotation_issues = []

for split in ("train", "val", "test"):
    image_dir = DATASET_DIR / "images" / split
    label_dir = DATASET_DIR / "labels" / split

    image_files = sorted(
        p for p in image_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ) if image_dir.exists() else []

    label_files = sorted(label_dir.rglob("*.txt")) if label_dir.exists() else []
    image_keys = {
        (p.relative_to(image_dir).parent, p.stem): p for p in image_files
    }
    label_keys = {
        (p.relative_to(label_dir).parent, p.stem): p for p in label_files
    }

    missing_labels = sorted(set(image_keys) - set(label_keys))
    orphan_labels = sorted(set(label_keys) - set(image_keys))
    empty_labels = 0
    valid_object_images = 0
    negative_images = 0
    invalid_label_files = 0
    split_counts = {int(k): 0 for k in class_names}

    for key, image_path in image_keys.items():
        label_path = label_keys.get(key)
        if label_path is None:
            # Ausência do TXT não é classificada como imagem negativa.
            continue

        lines = [
            line.strip() for line in label_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
        if not lines:
            empty_labels += 1
            negative_images += 1
            continue

        valid_in_file = 0
        for line_number, line in enumerate(lines, start=1):
            parts = line.split()
            try:
                if len(parts) != 5:
                    raise ValueError("esperados 5 valores")
                class_id = int(float(parts[0]))
                coords = np.asarray([float(v) for v in parts[1:]], dtype=float)
                if class_id not in class_names:
                    raise ValueError(f"class_id desconhecido: {class_id}")
                if not np.isfinite(coords).all():
                    raise ValueError("coordenadas não finitas")
                if (coords < 0).any() or (coords > 1).any():
                    raise ValueError("coordenadas fora do intervalo [0, 1]")
                if coords[2] <= 0 or coords[3] <= 0:
                    raise ValueError("largura/altura não positivas")
                split_counts[class_id] += 1
                class_distribution[class_id] += 1
                valid_in_file += 1
            except Exception as exc:
                invalid_label_files += 1
                annotation_issues.append({
                    "split": split, "arquivo": str(label_path),
                    "linha": line_number, "erro": str(exc)
                })

        if valid_in_file:
            valid_object_images += 1

    dataset_rows.append({
        "split": split,
        "imagens": len(image_files),
        "arquivos_txt": len(label_files),
        "imagens_sem_txt": len(missing_labels),
        "txt_sem_imagem": len(orphan_labels),
        "imagens_com_objetos_validos": valid_object_images,
        "imagens_negativas_txt_vazio": negative_images,
        "anotacoes_invalidas": invalid_label_files,
        **{f"caixas_{class_names[k]}": v for k, v in split_counts.items()}
    })

dataset_summary = pd.DataFrame(dataset_rows)
display(dataset_summary)

issues_df = pd.DataFrame(annotation_issues)
if not issues_df.empty:
    issues_df.to_csv(ARTIFACTS_DIR / "anotacoes_invalidas.csv", index=False)

if dataset_summary[["imagens", "arquivos_txt"]].eq(0).any().any():
    raise RuntimeError("Há um split sem imagens ou arquivos de anotação.")
if (dataset_summary["imagens_sem_txt"] > 0).any() or (dataset_summary["txt_sem_imagem"] > 0).any():
    raise RuntimeError(
        "Foram encontradas imagens sem TXT ou anotações órfãs. "
        "Corrija a correspondência antes do treinamento."
    )
if (dataset_summary["anotacoes_invalidas"] > 0).any():
    raise RuntimeError(
        "Foram encontradas anotações inválidas. Consulte anotacoes_invalidas.csv."
    )

class_distribution_df = pd.DataFrame([
    {"classe_id": k, "classe": class_names[k], "caixas": v}
    for k, v in class_distribution.items()
])
display(class_distribution_df)
dataset_summary.to_csv(ARTIFACTS_DIR / "auditoria_dataset.csv", index=False)
class_distribution_df.to_csv(ARTIFACTS_DIR / "distribuicao_classes.csv", index=False)

## 4. Configuração do treinamento e Data Augmentation

In [ ]:
# YOLO26n é o modelo Ultralytics mais recente disponível nesta versão.
BASE_MODEL = "yolo26n.pt"

EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 8
PATIENCE = 10
DEVICE = 0 if torch.cuda.is_available() else "cpu"
WORKERS = 0 if platform.system() == "Windows" else min(8, os.cpu_count() or 1)

# Augmentation moderado: preserva a geometria dos objetos e varia iluminação/cor.
AUGMENTATION = {
    "hsv_h": 0.015,
    "hsv_s": 0.50,
    "hsv_v": 0.30,
    "degrees": 3.0,
    "translate": 0.05,
    "scale": 0.30,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.50,
    "mosaic": 0.50,
    "mixup": 0.0,
    "copy_paste": 0.0,
    "close_mosaic": 10,
}

training_config = {
    "model": BASE_MODEL,
    "epochs": EPOCHS,
    "imgsz": IMAGE_SIZE,
    "batch": BATCH_SIZE,
    "patience": PATIENCE,
    "device": DEVICE,
    "workers": WORKERS,
    "seed": SEED,
    "pretrained": True,
    "augmentation": AUGMENTATION,
    "data_yaml": str(DATA_YAML.resolve()),
    "created_at": datetime.now().astimezone().isoformat(),
}
config_path = ARTIFACTS_DIR / "configuracao_treino.json"
config_path.write_text(
    json.dumps(training_config, indent=2, ensure_ascii=False), encoding="utf-8"
)
display(pd.DataFrame(
    [{"parametro": k, "valor": v} for k, v in AUGMENTATION.items()]
))

## 5. Registro do ambiente

In [ ]:
environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "processor": platform.processor(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "ultralytics": ultralytics.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "cuda_available": torch.cuda.is_available(),
}
if torch.cuda.is_available():
    environment["device_name"] = torch.cuda.get_device_name(0)

env_path = ARTIFACTS_DIR / "ambiente.json"
env_path.write_text(
    json.dumps(environment, indent=2, ensure_ascii=False), encoding="utf-8"
)
freeze = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    capture_output=True, text=True, check=False
)
(ARTIFACTS_DIR / "pip_freeze.txt").write_text(freeze.stdout, encoding="utf-8")
display(pd.DataFrame([environment]))

## 6. Fine-tuning

In [ ]:
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

rows = []
if RESULTS_CSV.exists():
    rows = pd.read_csv(RESULTS_CSV).to_dict("records")

training_completed = (
    len(rows) >= EPOCHS and BEST_WEIGHTS.exists()
)

if training_completed:
    print(f"Treinamento já concluído: {len(rows)}/{EPOCHS} épocas.")
    model = YOLO(str(BEST_WEIGHTS))

elif LAST_WEIGHTS.exists():
    checkpoint = torch.load(
        LAST_WEIGHTS, map_location="cpu", weights_only=False
    )
    resumable = (
        int(checkpoint.get("epoch", -1)) >= 0
        and checkpoint.get("optimizer") is not None
    )
    if not resumable:
        raise RuntimeError(
            "O checkpoint existente não permite retomada. "
            "Defina NEW_VERSION=True para iniciar uma nova execução preservando esta."
        )
    print("Retomando treinamento a partir do último checkpoint.")
    model = YOLO(str(LAST_WEIGHTS))
    model.train(resume=True)

else:
    print("Iniciando fine-tuning com augmentation.")
    model = YOLO(BASE_MODEL)
    model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        patience=PATIENCE,
        device=DEVICE,
        workers=WORKERS,
        seed=SEED,
        deterministic=True,
        pretrained=True,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        plots=True,
        save=True,
        verbose=True,
        **AUGMENTATION,
    )

if not BEST_WEIGHTS.exists():
    raise FileNotFoundError(f"Pesos best.pt não encontrados em {BEST_WEIGHTS}")

# Recarrega o histórico após treino/retomada.
if RESULTS_CSV.exists():
    results_df = pd.read_csv(RESULTS_CSV)
    print("Épocas registradas:", len(results_df))
    print("Última época registrada:", int(results_df["epoch"].iloc[-1]) + 1)
else:
    raise FileNotFoundError(f"Histórico não encontrado: {RESULTS_CSV}")

model = YOLO(str(BEST_WEIGHTS))

## 7. Curvas e métricas por época

In [ ]:
results_df = pd.read_csv(RESULTS_CSV)
results_df.columns = [c.strip() for c in results_df.columns]
results_df.to_csv(ARTIFACTS_DIR / "results.csv", index=False)

loss_cols = [
    c for c in results_df.columns
    if "loss" in c.lower() and pd.api.types.is_numeric_dtype(results_df[c])
]
metric_cols = [
    c for c in results_df.columns
    if c.startswith("metrics/") and pd.api.types.is_numeric_dtype(results_df[c])
]

if loss_cols:
    ax = results_df.plot(x="epoch", y=loss_cols, figsize=(11, 5))
    ax.set_title("Loss por época")
    ax.set_xlabel("Época")
    ax.set_ylabel("Loss")
    ax.grid(True)
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / "curvas_loss.png", dpi=150)
    plt.show()

if metric_cols:
    ax = results_df.plot(x="epoch", y=metric_cols, figsize=(11, 5))
    ax.set_title("Métricas de validação por época")
    ax.set_xlabel("Época")
    ax.set_ylabel("Valor")
    ax.grid(True)
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / "curvas_metricas.png", dpi=150)
    plt.show()

display(results_df.tail(10))

## 8. Avaliação no conjunto de teste

In [ ]:
test_metrics = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True,
    project=str(RUNS_DIR),
    name=f"{RUN_NAME}-test",
    exist_ok=True,
)

box = test_metrics.box
metrics_summary = {
    "experimento": EXPERIMENT_NAME,
    "modelo": BASE_MODEL,
    "split": "test",
    "precision": float(box.mp),
    "recall": float(box.mr),
    "mAP50": float(box.map50),
    "mAP50_95": float(box.map),
    "fitness": float(box.fitness()) if hasattr(box, "fitness") else None,
    "weights": str(BEST_WEIGHTS.resolve()),
    "evaluated_at": datetime.now().astimezone().isoformat(),
}
metrics_path = ARTIFACTS_DIR / "metricas_teste.json"
metrics_path.write_text(
    json.dumps(metrics_summary, indent=2, ensure_ascii=False), encoding="utf-8"
)
display(pd.DataFrame([metrics_summary]))

## 9. Métricas por classe

In [ ]:
test_label_dir = DATASET_DIR / "labels" / "test"
class_counts = {int(k): 0 for k in class_names}
for label_file in test_label_dir.rglob("*.txt"):
    for line in label_file.read_text(encoding="utf-8").splitlines():
        if line.strip():
            class_id = int(line.split()[0])
            class_counts[class_id] = class_counts.get(class_id, 0) + 1

per_class_rows = []
for index, class_id in enumerate(box.ap_class_index):
    class_id = int(class_id)
    per_class_rows.append({
        "classe_id": class_id,
        "classe": class_names.get(class_id, str(class_id)),
        "precision": float(box.p[index]) if len(box.p) > index else np.nan,
        "recall": float(box.r[index]) if len(box.r) > index else np.nan,
        "AP50": float(box.ap50[index]),
        "AP50-95": float(box.ap[index]),
        "caixas_reais_no_teste": class_counts.get(class_id, 0),
    })

per_class_df = pd.DataFrame(per_class_rows)
display(per_class_df)
per_class_df.to_csv(ARTIFACTS_DIR / "metricas_por_classe.csv", index=False)

## 10. Registro final da execução

In [ ]:
report = {
    "experiment": EXPERIMENT_NAME,
    "created_at": datetime.now().astimezone().isoformat(),
    "configuration": training_config,
    "environment": environment,
    "dataset_audit": dataset_rows,
    "class_distribution": class_distribution,
    "test_metrics": metrics_summary,
    "per_class_metrics": per_class_rows,
    "artifacts": {
        "training_results": str((ARTIFACTS_DIR / "results.csv").resolve()),
        "test_metrics": str(metrics_path.resolve()),
        "per_class_metrics": str((ARTIFACTS_DIR / "metricas_por_classe.csv").resolve()),
        "best_weights": str(BEST_WEIGHTS.resolve()),
        "last_weights": str(LAST_WEIGHTS.resolve()),
    },
}
report_path = ARTIFACTS_DIR / "registro_experimento.json"
report_path.write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
)

print("Execução:", EXPERIMENT_NAME)
print("Modelo:", BASE_MODEL)
print("Melhores pesos:", BEST_WEIGHTS)
print("Métricas de teste:")
display(pd.DataFrame([metrics_summary]))
print("Métricas por classe:")
display(per_class_df)